# LLM 수동 입력 테스트

저장소 루트에서 `uv run --locked jupyter lab ai/app/test.ipynb`로 엽니다.
분류 모델을 사용하지 않습니다. 분야와 질문을 직접 입력합니다.
실제 EXAONE 가중치와 세 어댑터가 필요하며, 최초 실행 시 모델을 다운로드할 수 있습니다.
CPU는 메모리와 생성 시간이 많이 필요합니다. 장치 설정을 바꾼 뒤에는 커널을 재시작하세요.

In [ ]:
import os
import sys
from pathlib import Path
from time import perf_counter

# CPU로 강제하려면 아래 두 줄의 주석을 해제하세요.
# os.environ['LLM_DEVICE'] = 'cpu'
# os.environ['LLM_LOAD_IN_4BIT'] = 'false'

for parent in (Path.cwd(), *Path.cwd().parents):
    ai_root = parent if (parent / 'app' / 'llm').is_dir() else parent / 'ai'
    if (ai_root / 'app' / 'llm').is_dir():
        sys.path.insert(0, str(ai_root))
        break
else:
    raise FileNotFoundError('dontalk 저장소 안에서 노트북을 실행하세요.')

from app.core import config
from app.llm.model import answer, load_model

print(f'설정: device={config.LLM_DEVICE}, 4bit={config.LLM_LOAD_IN_4BIT}')

In [ ]:
started = perf_counter()
model, tokenizer = load_model()
print(f'모델 준비: {perf_counter() - started:.2f}초')
print(f'실제 장치: {model.get_input_embeddings().weight.device}')
assert load_model()[0] is model  # 같은 커널에서는 재로드하지 않습니다.

# RAG 임시 연결

In [ ]:
import os
import psycopg2
from psycopg2.extras import RealDictCursor
from pgvector.psycopg2 import register_vector
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv(override=True)

In [ ]:
POSTGRESQL_PORT = os.getenv("DB_PORT")
POSTGRESQL_USER = os.getenv("DB_USER")
POSTGRESQL_PASSWORD = os.getenv("DB_PASSWORD")
POSTGRESQL_DBNAME = os.getenv("DB_NAME")
POSTGRESQL_HOST = os.getenv("DB_HOST")

In [ ]:

DB_CONFIG = {
    "host": POSTGRESQL_HOST,
    "port": POSTGRESQL_PORT,
    "user": POSTGRESQL_USER,
    "password": POSTGRESQL_PASSWORD,
    "dbname": POSTGRESQL_DBNAME
}

conn = psycopg2.connect(**DB_CONFIG ,sslmode="require", sslnegotiation="direct", connect_timeout=10)
register_vector(conn)
cur = conn.cursor(cursor_factory=RealDictCursor)

rag_model = SentenceTransformer("dragonkue/snowflake-arctic-embed-l-v2.0-ko")
print("DB 연결 및 모델 로드 완료")

In [ ]:
# --------------------------------------------------------
# RAG TEST - 코사인 유사도 기준 Top-3
# --------------------------------------------------------
def search_top3(user_query: str, embedding_column: str = "embedding_q"):
    # user_query: 사용자가 입력한 신규 질의 텍스트
    # embedding_column: embedding_q / embedding_i / embedding_full 중 선택

    # 1) 신규 질의 인코딩 (query 프롬프트 적용 - 저장된 데이터와 다르게 처리해야 함)
    query_vector = rag_model.encode(user_query, prompt_name="query", normalize_embeddings=True)

    # 2) 코사인 거리 기준 Top-3 검색
    sql = f"""
        SELECT qa_id, full_source, consulting_category, consulting_topic, qa_topic, consulting_purpose,
               {embedding_column} <=> %s AS distance
        FROM financial_consulting_qa
        ORDER BY {embedding_column} <=> %s
        LIMIT 3;
    """
    cur.execute(sql, (query_vector, query_vector))
    return cur.fetchall()

def print_results(results):
    for i, row in enumerate(results, 1):
        print("---------------------------------------------------")
        print(f"[{i}순위] 거리: {row['distance']:.4f}")
        print(f"과거 질의응답:\n{row['full_source']}")
        print(f"metadata:\n- consulting_category={row['consulting_category']}\n- consulting_topic={row['consulting_topic']}\n- qa_topic={row['qa_topic']}\n- consulting_purpose={row['consulting_purpose']}")
        print("---------------------------------------------------")

def most_category(results):
    """최빈 분야를 반환한다. 동률이면 검색 순위가 높은 분야, 유효한 값이 없으면 None."""
    from collections import Counter

    counts = Counter(row["consulting_category"] for row in results if row.get("consulting_category"))
    return counts.most_common(1)[0][0] if counts else None


1. RAG랑 같이

In [ ]:
MAX_NEW_TOKENS = 512

while True:
    question = input('질문, 끝내려면 종료:: ').strip()
    if question == '종료':
        break
    started = perf_counter()

    rag_docs = search_top3(question)
    print(f'RAG 탐색: {perf_counter() - started:.2f}초\n')
    print(rag_docs)

    category = most_category(rag_docs)
    m = perf_counter()
    llm_input = f"고객 질문 : {question}\nRAG 결과: {rag_docs}"

    try:
        result = answer(category, llm_input, max_new_tokens=MAX_NEW_TOKENS)
    except ValueError as error:
        print(error)
        continue

    print(f"\n 질문 : {question}")
    print(f"\n[{category}] {result}")
    print(f'답변 생성: {perf_counter() - m:.2f}초\n')

2. 그냥 모델로

In [ ]:
MAX_NEW_TOKENS = 512

while True:

    category = input('분야 [보험, 증권, 은행], 끝내려면 종료:: ').strip()
    if category == '종료':
        break

    question = input('질문, 끝내려면 종료:: ').strip()
    if question == '종료':
        break
    started = perf_counter()
    try:
        result = answer(category, question, max_new_tokens=MAX_NEW_TOKENS)
    except ValueError as error:
        print(error)
        continue

    print(f"\n 질문 : {question}")
    print(f"\n[{category}] {result}")
    print(f'답변 생성: {perf_counter() - started:.2f}초\n')